# Original course-project exploration
Spring 2025 team project. Hongjia Liu was the primary contributor to implementation and experiments.
This notebook preserves historical code with outputs cleared. It is not the revised pipeline.
Known evaluation limitations and required external data/caches are described in ../docs/PROJECT_REPORT.md.
Run revised experiments from src/pipeline.py at the repository root.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# -------------------------------------------------------------
# EmoEEG-MC  三分类 · 被试内 10-fold CV · 单频段 (4 s × 4 帧)
# -------------------------------------------------------------
import os, glob, math, random, itertools
from pathlib import Path
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from tqdm import tqdm
import torch, torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from model.SWIN_trans import SwinTransformer     # 论文实现
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:

# -------------------------------------------------------------
# 1. 改这里：目录 / 训练参数 / 场景 / 频段
# -------------------------------------------------------------
DATA_DIR   = './data'          # 原始 EmoEEG-MC
CACHE_DIR  = './cache3k1s_all'     # 自动生成
RUNS_DIR   = './runs_3k'          # 结果

# -------------------------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
Path(CACHE_DIR).mkdir(exist_ok=True)
Path(RUNS_DIR).mkdir(exist_ok=True)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
torch.backends.cudnn.benchmark = True

# -------------------------------------------------------------
# 2. 通道 → 9×9 网格
# -------------------------------------------------------------
os.makedirs(CACHE_DIR, exist_ok=True)
CH_NAMES = ['Fp1','Fpz','Fp2','AF7','AF3','AF4','AF8',
            'F7','F5','F3','F1','Fz','F2','F4','F6','F8',
            'FT7','FC5','FC3','FC1','FCz','FC2','FC4','FC6','FT8',
            'T7','C5','C3','C1','Cz','C2','C4','C6','T8',
            'TP7','CP5','CP3','CP1','CPz','CP2','CP4','CP6','TP8',
            'P7','P5','P3','P1','Pz','P2','P4','P6','P8',
            'PO7','PO3','POz','PO4','PO8','O1','Oz','O2','F9','F10','TP9','TP10']

# 手动把 64 电极放进 9×9，示例只列两行——生产环境请补全
GRID9 = np.full((9, 9), None, dtype=object)      # 先全部置空

# 每行 list = {(row, col): 'ChanName'}
coords = {
    # row 0  ──────────────────────────────────────────────
    'F9':  (0,0),  'Fp1':(0,2),  'Fpz':(0,4),  'Fp2':(0,6),  'F10':(0,8),

    # row 1
    'AF7': (1,1),  'AF3':(1,3),  'AF4':(1,5),  'AF8':(1,7),

    # row 2
    'F7':  (2,0),  'F5': (2,1),  'F3': (2,2),  'F1': (2,3),  'Fz': (2,4),
    'F2':  (2,5),  'F4': (2,6),  'F6': (2,7),  'F8': (2,8),

    # row 3
    'FT7': (3,0),  'FC5':(3,1),  'FC3':(3,2),  'FC1':(3,3),  'FCz':(3,4),
    'FC2': (3,5),  'FC4':(3,6),  'FC6':(3,7),  'FT8':(3,8),

    # row 4
    'T7':  (4,0),  'C5': (4,1),  'C3': (4,2),  'C1': (4,3),  'Cz': (4,4),
    'C2':  (4,5),  'C4': (4,6),  'C6': (4,7),  'T8': (4,8),

    # row 5
    'TP7': (5,0),  'CP5':(5,1),  'CP3':(5,2),  'CP1':(5,3),  'CPz':(5,4),
    'CP2': (5,5),  'CP4':(5,6),  'CP6':(5,7),  'TP8':(5,8),

    # row 6
    'P7':  (6,0),  'P5': (6,1),  'P3': (6,2),  'P1': (6,3),  'Pz': (6,4),
    'P2':  (6,5),  'P4': (6,6),  'P6': (6,7),  'P8': (6,8),

    # row 7
    'PO7': (7,1),  'PO3':(7,3),  'POz':(7,4),  'PO4':(7,5),  'PO8':(7,7),

    # row 8
    'TP9': (8,0),  'O1': (8,2),  'Oz': (8,4),  'O2': (8,6),  'TP10':(8,8),
}
idx2rc   = {CH_NAMES.index(ch): rc for ch, rc in coords.items()}


In [ ]:
# BAND = 4
def topo_map(mat64x4):
    """64×4 → (4,32,32)"""
    grid9 = np.zeros((9,9,4), np.float32)
    for ch,(r,c) in idx2rc.items():
        grid9[r,c] = mat64x4[ch]
    interp = RegularGridInterpolator((range(9),)*2, grid9,
                                     method='cubic', bounds_error=False, fill_value=0.)
    Y,X = np.mgrid[0:8:32j, 0:8:32j]
    return interp(np.stack([Y,X],-1)).transpose(2,0,1)   # (4,32,32)

LBL7_TO_3 = np.array([0,0,0,1,2,2,2], np.int64)  
# -------------------------------------------------------------
# 3. 预处理：30 s → 7 个 4 s 片段
# -------------------------------------------------------------
def preprocess_subject(sub:int, ses:str,
                       win_sec:int=4, stride_sec:int=4) -> None:
    fn_out = Path(CACHE_DIR) / f'sub{sub:02d}_{ses}.npz'
    if fn_out.exists():
        print(f'[✓] {fn_out.name} 已存在，跳过')
        return

    fn_in  = Path(DATA_DIR) / f'sub-{sub:02d}' / f'sub-{sub:02d}_ses-{ses}_task-emotion_de.npy'
    if not fn_in.exists():            # 原文件缺失
        print(f'[!] 缺少 {fn_in}')
        return

    de = np.load(fn_in, mmap_mode='r').astype(np.float32)    # (64,630,5)
    de = de.reshape(64,21,30,5)                              # (chan, trial, sec, band)

    X_band = [[] for _ in range(5)]                          # 5 个容器
    y      = []

    for t in range(21):
        cls3 = LBL7_TO_3[t//3]
        for beg in range(0,28,4):                            # 7 个 4-s 片段
            for b in range(5):
                slice4 = de[:,t,beg:beg+4,b]                 # (64,4)
                X_band[b].append(topo_map(slice4))           # (4,32,32)
            y.append(cls3)

    # 堆叠 & 保存
    X_band = [np.stack(lst).astype(np.float32) for lst in X_band]
    y      = np.asarray(y, dtype=np.int64)
    np.savez_compressed(fn_out,
                        x0=X_band[0], x1=X_band[1], x2=X_band[2],
                        x3=X_band[3], x4=X_band[4], y=y)
    shapes = ' '.join([f'x{b}{X_band[b].shape}' for b in range(5)])
    print(f'[+] 保存 {fn_out.name}: {shapes} y{y.shape}')



In [ ]:

# -------------------------------------------------------------
# 4. 单被试数据集
# -------------------------------------------------------------
class EmoEEGDataset(Dataset):
    def __init__(self, sub:int, context:str, band:int):
        patt  = f'sub{sub:02d}_*.npz' if context=='multi' else f'sub{sub:02d}_{context}.npz'
        files = glob.glob(str(Path(CACHE_DIR)/patt))
        xs, ys = [], []
        for f in files:
            z = np.load(f)
            xs.append(z[f'x{band}'])           # <- 关键改动
            ys.append(z['y'])
        self.x = np.concatenate(xs).astype(np.float32)
        self.y = np.concatenate(ys).astype(np.int64)
        self.mu = self.x.mean((0,2,3), keepdims=True)
        self.sd = self.x.std ((0,2,3), keepdims=True)+1e-8

    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = (self.x[idx] - self.mu)/self.sd
        x = x.squeeze()
        return torch.from_numpy(x), int(self.y[idx])

In [ ]:

# -------------------------------------------------------------
# 5. Swin-Transformer (in_chans=4‼)
# -------------------------------------------------------------

def build_swin(in_chans=4, num_classes=3):
    return SwinTransformer(
            # Model parameter
            img_size=32,
            patch_size=4,
            in_chans=in_chans, num_classes=num_classes,
            embed_dim=96,
            depths=[2, 2, 6, 2],  
            num_heads=[3, 6, 12, 24],
            window_size=2,
            mlp_ratio=4., qkv_bias=True,
            drop_rate=0.17, attn_drop_rate=0.17, drop_path_rate=0.1,
            patch_norm=True,
            use_checkpoint=False
    )

CLS_W = torch.tensor([1.,2.,1.], device=device)


In [ ]:

def cm2percent(cm: np.ndarray, axis=None) -> np.ndarray:
    div = cm.sum(axis=axis, keepdims=True)
    div[div == 0] = 1
    return cm / div * 100

def plot_cm(cm: np.ndarray, title: str, fname: Path, as_percent=True):
    disp = cm2percent(cm) if as_percent else cm
    fmt  = '.1f' if as_percent else 'd'
    sns.heatmap(disp, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, cbar=False)
    plt.title(title); plt.tight_layout()
    fname.parent.mkdir(exist_ok=True, parents=True)
    plt.savefig(fname, dpi=120); plt.close()

In [ ]:
BAND_NAMES = ['delta', 'theta', 'alpha', 'beta', 'gamma']
# -------------------------------------------------------------
# 6. 训练单被试
# -------------------------------------------------------------
from sklearn.metrics import confusion_matrix

BAND_NAMES = ['delta', 'theta', 'alpha', 'beta', 'gamma']
# 6. 训练单被试（保持原样，只删去无关 import）
def train_one_subject(sub: int, band:int):
    ds  = EmoEEGDataset(sub, CTX,band)
    kf  = KFold(n_splits=5, shuffle=True, random_state=42)

    best_acc_sub, best_cm_sub = 0.0, None
    print(f'\n======= Subject {sub:02d} =======')
    for k, (tr_idx, te_idx) in enumerate(kf.split(ds), 1):
        tr, te = Subset(ds, tr_idx), Subset(ds, te_idx)
        tr_ld  = DataLoader(tr, BATCH_TR, shuffle=True, pin_memory=True)
        te_ld  = DataLoader(te, BATCH_TE, pin_memory=True)

        net = build_swin().to(device)
        head, back = [], []
        for n, p in net.named_parameters():
            (head if n.startswith('head') else back).append(p)
        opt = torch.optim.AdamW(
                [{'params': back, 'lr': 1e-4},
                 {'params': head, 'lr': 5e-4}],
                weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.LambdaLR(
                opt, lr_lambda=lambda ep: (1.0 if ep < 100 else
                                           0.1 if ep < 150 else 0.01))
        best_acc_fold, best_cm_fold = 0, None

        for ep in range(1, EPOCHS + 1):
            # ---- train ----
            net.train(); loss_tot = 0
            for x, y in tr_ld:
                x, y = x.to(device), y.to(device)
                loss = F.cross_entropy(net(x), y, weight=CLS_W)
                opt.zero_grad(); loss.backward(); opt.step()
                loss_tot += loss.item()
            # ---- val ----
            net.eval(); preds, gts = [], []
            with torch.no_grad():
                for x, y in te_ld:
                    p = net(x.to(device)).argmax(1).cpu()
                    preds.append(p); gts.append(y)
            preds, gts = torch.cat(preds), torch.cat(gts)
            acc = (preds == gts).float().mean().item()
            scheduler.step()

            if acc > best_acc_fold:
                best_acc_fold = acc
                best_cm_fold  = confusion_matrix(gts, preds, labels=[0, 1, 2])

            if ep % 5 == 0:
                print(f'[{CTX}-{BAND_NAMES[BAND]}] S{sub:02d} F{k} '
                      f'Ep{ep:03d} loss={loss_tot/len(tr_ld):.4f} acc={acc:.3f}')

        # ---- fold CM ----
        plot_cm(best_cm_fold,
                title=f'S{sub:02d} F{k} best acc={best_acc_fold:.3f}',
                fname=RUNS_DIR / 'cm' / f's{sub:02d}_f{k}.png')

        if best_acc_fold > best_acc_sub:
            best_acc_sub, best_cm_sub = best_acc_fold, best_cm_fold

    print(f'>> S{sub:02d}  best acc = {best_acc_sub:.3f}')
    # 保存各 subject 最佳 CM、acc
    np.save(RUNS_DIR / f'cm_sub{sub:02d}.npy', best_cm_sub)
    with open(RUNS_DIR / 'sub_accs.txt', 'a') as f:
        f.write(f'{sub:02d},{best_acc_sub:.4f}\n')
    plot_cm(best_cm_sub,
            title=f'S{sub:02d}  best acc={best_acc_sub:.3f}',
            fname=RUNS_DIR / f'cm_sub{sub:02d}.png')
    return best_cm_sub, best_acc_sub



In [ ]:
LABELS   = ['neg', 'neu', 'pos']
CTX_OPTIONS   = ['vid', 'ima', 'multi']
BAND_OPTIONS = list(range(5))
SUBJECTS   = range(15,17)            # ⇽ 需要的话改成 1-54
EPOCHS     = 160
BATCH_TR   = BATCH_TE = 64
CTX        = 'multi'           # 'ima' | 'vid' | 'multi'
# BAND       = 3                 # 0-δ 1-θ 2-α 3-β 4-γ
if isinstance(RUNS_DIR, str):
    RUNS_DIR = Path(RUNS_DIR)
if __name__ == '__main__':
    # ---------------------------------------------------------
    # 1) 逐 (情境, 频段) 训练 & 收集数据
    # ---------------------------------------------------------
    acc_dict = {ctx: {band: [] for band in BAND_OPTIONS} for ctx in CTX_OPTIONS}
    best_cm_each_subject = []      # 每个受试者 global 最佳 CM

    for sid in SUBJECTS:
        best_acc_person, best_cm_person = 0, None

        for ctx in CTX_OPTIONS:
            CTX = ctx              # 仍用全局变量，不改 train_one_subject
            for band in BAND_OPTIONS:
                BAND = band
                cm, acc = train_one_subject(sid,band)
                acc_dict[ctx][band].append(acc)

                if acc > best_acc_person:
                    best_acc_person, best_cm_person = acc, cm

        best_cm_each_subject.append(best_cm_person)


In [ ]:

    # ---------------------------------------------------------
    # 2) 画柱状图：五频段 × 三情境
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 4))
    width   = 0.2
    x_pos   = np.arange(len(BAND_OPTIONS))     # 0…4
    
    CTX_COLORS = {
        'vid'  : '#c7e3c1',   # 取样的淡绿色
        'ima'  : '#f4b082',   # 取样的淡橙色
        'multi': '#d2d9f5'    # 自配的淡蓝，区分第三条
    }

    for i, ctx in enumerate(CTX_OPTIONS):
        means = [np.mean(acc_dict[ctx][b]) for b in BAND_OPTIONS]
        ax.bar(x_pos + (i-1)*width, means, width,
            label=ctx,
            color=CTX_COLORS[ctx], 
            edgecolor='black', linewidth=0.8)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(BAND_NAMES)
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Frequency Band')
    ax.set_ylim(0, 1.0)
    ax.axhline(0.333, ls='--', c='k', lw=1, label='Chance 33%')
    ax.legend()
    ax.set_title('Ternary Classification Within Subject')
    plt.tight_layout()
    plt.savefig(RUNS_DIR / 'bar_best_acc.png', dpi=120); plt.close()

    # ---------------------------------------------------------
    # 3) 汇总最佳 CM → 百分比图
    # ---------------------------------------------------------
    grand_best_cm = sum(best_cm_each_subject)
    mean_best_acc = np.mean([np.max([acc_dict[ctx][b][i]
                        for ctx in CTX_OPTIONS for b in BAND_OPTIONS])
                        for i, _ in enumerate(SUBJECTS)])

    plot_cm(grand_best_cm,
            title=f'Overall confusion matrix  (mean best acc={mean_best_acc:.3f})',
            fname=RUNS_DIR / 'cm_overall_best.png',
            as_percent=True)

    print('\n========== FINISHED ==========')
    for ctx in CTX_OPTIONS:
        for b in BAND_OPTIONS:
            print(f'{ctx}-{BAND_NAMES[b]}: {np.mean(acc_dict[ctx][b]):.3f}')
    print(f'总体平均最佳 acc: {mean_best_acc:.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

# ── 1. 假设 acc_dict / best_cm_each_subject 仍在内存 ─────────────
#     S15 是第一个完成的 subject ⇒ 索引 0
idx_s15   = 0
band_ids  = list(range(5))
band_names = ['δ','θ','α','β','γ']
ctxs      = ['vid', 'ima', 'multi']
ctx_colors = {'vid':'#c7e3c1','ima':'#f4b082','multi':'#d2d9f5'}

# ── 2. 画五频段 × 三情境柱状图 ────────────────────────────────
fig, ax = plt.subplots(figsize=(8,4))
width   = 0.22
x_pos   = np.arange(len(band_ids))

for j, ctx in enumerate(ctxs):
    vals = [acc_dict[ctx][b][idx_s15] for b in band_ids]
    ax.bar(x_pos + (j-1)*width, vals, width,
           label=ctx, color=ctx_colors[ctx],
           edgecolor='black', linewidth=0.8)
    for x, v in zip(x_pos + (j-1)*width, vals):
        ax.text(x, v+0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(band_names)
ax.set_ylabel('Accuracy')
ax.set_xlabel('Frequency Band')
ax.set_ylim(0, 1.0)
ax.axhline(1/3, ls='--', c='k', lw=1, label='Chance 33 %')
ax.legend()
ax.set_title('Subject 15 • Five Bands × Three Contexts')
plt.tight_layout()
Path('plots').mkdir(exist_ok=True)
plt.savefig('plots/bar_S15.png', dpi=120)
plt.close()

# ── 3. 画 Subject 15 的最佳混淆矩阵 ─────────────────────────
cm_s15 = best_cm_each_subject[idx_s15]

def cm2percent(cm, axis=None):
    div = cm.sum(axis=axis, keepdims=True)
    div[div==0] = 1
    return cm / div * 100

disp = cm2percent(cm_s15)
sns.heatmap(disp, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=['neg','neu','pos'],
            yticklabels=['neg','neu','pos'],
            cbar=False)
plt.title('Subject 15 • Confusion Matrix (%)')
plt.tight_layout()
plt.savefig('plots/cm_S15.png', dpi=120)
plt.close()
